# Fine-tuning de embeddings · dominio legal

> Un modelo de embeddings genérico no entiende la jerga legal: el usuario pregunta en lenguaje
> llano ("¿pueden echarme del piso?") y el documento está en jerga ("resolución del contrato de
> arrendamiento, desahucio..."). Lo afinamos con pares (pregunta, fragmento legal) y medimos
> el retrieval **antes vs después**. Mismo arco que en el módulo de Fine-tuning (clase 3): medir, entrenar, medir.

## Qué aprendemos

1. Por qué un embedding genérico falla en un dominio especializado (el hueco llano <-> jerga).
2. **Fine-tuning contrastivo** con `sentence-transformers` y `MultipleNegativesRankingLoss`
   (acerca la pregunta y su documento relevante en el espacio vectorial).
3. Medir el retrieval con un número: **top-1 accuracy** y **MRR**, antes vs después.

## Por qué importa

El retrieval es el cuello de botella de un RAG (lo vemos en el módulo de RAG): si el buscador no encuentra el documento
correcto, el LLM no puede responder bien. Afinar el embedding a TU dominio es la palanca más barata
para subir la calidad del RAG.

## Stack

`sentence-transformers` (v6+) · `datasets` · modelo base multilingüe pequeño.

## Hardware

T4 (Colab Free) de sobra; el modelo es pequeño (~118M) y corre incluso en CPU o en un Mac (MPS):
el entrenamiento tarda unos 18 s en un Mac M-series (medido el 2026-10-14).


In [1]:
!pip install -qU "sentence-transformers>=6" datasets

## 1 · Un mini-corpus legal (pregunta en llano -> fragmento en jerga)

Pares `(pregunta, fragmento)` donde la pregunta está en lenguaje de calle y el fragmento en jerga
jurídica. Justo el hueco que un embedding genérico no cubre bien.

> Los textos son **ilustrativos y didácticos** (no son asesoramiento legal ni citan artículos
> concretos). Lo que importa aquí es el patrón llano <-> jerga, no la exactitud normativa.


In [2]:
PARES = [
    ("¿Pueden echarme del piso de alquiler sin avisar?",
     "El arrendador no puede instar la resolucion del contrato de arrendamiento sin la notificacion fehaciente y el preaviso legalmente exigible; el desahucio requiere resolucion judicial."),
    ("¿Cuanto tiempo tengo para reclamar una deuda?",
     "Las acciones personales sin plazo especial estan sometidas a un plazo de prescripcion, transcurrido el cual la obligacion deviene inexigible por el transcurso del tiempo."),
    ("Me han despedido, ¿cuanto me tienen que pagar?",
     "La extincion del contrato por causas objetivas o por despido improcedente genera derecho a una indemnizacion calculada sobre el salario regulador y la antiguedad del trabajador."),
    ("¿Que pasa con las deudas de alguien que muere?",
     "El llamado a la herencia puede aceptarla a beneficio de inventario, de modo que responde de las cargas y deudas hereditarias solo hasta donde alcancen los bienes del caudal relicto."),
    ("¿Me pueden grabar con camaras en el trabajo?",
     "El tratamiento de datos mediante videovigilancia exige una base de legitimacion, el deber de informacion al interesado y el respeto a los principios de minimizacion y proporcionalidad."),
    ("Compre algo roto por internet, ¿lo puedo devolver?",
     "El consumidor dispone de un derecho de desistimiento en la contratacion a distancia, asi como de las acciones de saneamiento por falta de conformidad del producto."),
    ("¿Que es la letra pequena de un contrato?",
     "Las condiciones generales de la contratacion se integran en el contrato de adhesion; las clausulas abusivas se reputan nulas de pleno derecho y se tienen por no puestas."),
    ("Mi vecino hace mucho ruido por la noche, ¿que hago?",
     "Las inmisiones que excedan la normal tolerancia facultan al perjudicado para ejercitar la accion de cesacion y, en su caso, reclamar la responsabilidad civil por los danos causados."),
    ("¿Puedo dejar mi herencia a quien yo quiera?",
     "La libertad de testar esta limitada por las legitimas, porcion de la herencia reservada por ministerio de la ley a los herederos forzosos."),
    ("Tuve un accidente y no fue mi culpa, ¿quien paga?",
     "Quien por accion u omision causa dano a otro interviniendo culpa o negligencia queda obligado a reparar el dano causado mediante la correspondiente indemnizacion."),
    ("¿Que firmo cuando monto una empresa con un socio?",
     "La constitucion de una sociedad mercantil exige escritura publica e inscripcion en el Registro Mercantil, que determinan el nacimiento de la personalidad juridica."),
    ("Me reclaman una multa muy antigua, ¿es legal?",
     "La potestad sancionadora esta sujeta a plazos de prescripcion de las infracciones y de las sanciones, cuyo transcurso extingue la responsabilidad."),
    ("¿Puedo grabar una conversacion en la que participo?",
     "La grabacion de comunicaciones por uno de los intervinientes no vulnera el secreto de las comunicaciones, sin perjuicio de los limites del derecho a la intimidad de terceros."),
    ("Mi jefe no me paga las horas extra, ¿que puedo hacer?",
     "El trabajador puede reclamar las cantidades adeudadas en concepto de salario mediante la accion de reclamacion de cantidad ante la jurisdiccion social."),
    ("¿Que pasa si no pago el alquiler un mes?",
     "El impago de la renta es causa de resolucion del contrato de arrendamiento y habilita el ejercicio de la accion de desahucio por falta de pago."),
    ("¿Puedo echarme atras tras firmar la compra de un coche?",
     "Perfeccionado el contrato de compraventa por el concurso de oferta y aceptacion, las partes quedan obligadas a su cumplimiento conforme a lo pactado."),
    ("¿Quien se queda la casa en un divorcio?",
     "La atribucion del uso de la vivienda familiar se acuerda atendiendo al interes mas necesitado de proteccion, sin prejuzgar la titularidad dominical del inmueble."),
    ("Firme como avalista de un amigo, ¿me pueden reclamar a mi?",
     "El fiador responde del cumplimiento de la obligacion principal; el acreedor puede dirigirse contra el fiador en los terminos de la fianza pactada."),
    ("¿Me pueden cobrar intereses altisimos por un prestamo?",
     "Los intereses notablemente superiores al normal del dinero y manifiestamente desproporcionados pueden reputarse usurarios y conllevar la nulidad del prestamo."),
    ("¿Que derechos tengo si compro un piso con desperfectos ocultos?",
     "El vendedor responde de los vicios o defectos ocultos de la cosa vendida que la hagan impropia para su uso, mediante las acciones edilicias."),
    ("¿Puedo despedir a una trabajadora embarazada?",
     "El despido de la trabajadora durante el periodo de proteccion por maternidad se declara nulo salvo que se acredite su procedencia por motivos ajenos al embarazo."),
    ("¿Que pasa si incumplo un contrato que firme?",
     "El incumplimiento de las obligaciones contractuales faculta a la parte cumplidora para exigir el cumplimiento forzoso o la resolucion, con resarcimiento de danos y perjuicios."),
    ("¿Puedo reclamar si un producto me causa una lesion?",
     "El fabricante responde de los danos causados por los defectos de sus productos conforme al regimen de responsabilidad civil por productos defectuosos."),
    ("¿Me pueden subir el alquiler cuando quieran?",
     "La actualizacion de la renta se rige por lo pactado y por los limites de revision previstos legalmente para los contratos de arrendamiento de vivienda."),
    ("¿Que es un poder notarial?",
     "El apoderamiento es el negocio juridico por el que se confiere a otro la representacion para actuar en nombre y por cuenta del poderdante."),
    ("¿Puedo recuperar el dinero de una compra fraudulenta con tarjeta?",
     "El titular de un instrumento de pago tiene derecho a la rectificacion de las operaciones no autorizadas, respondiendo el proveedor de servicios de pago salvo fraude o negligencia grave."),
    ("¿Tengo que pagar a Hacienda por una herencia?",
     "La adquisicion de bienes y derechos por herencia, legado o cualquier titulo sucesorio constituye el hecho imponible del impuesto sobre sucesiones."),
    ("¿Puedo dejar de pagar la hipoteca si entrego la casa?",
     "La dacion en pago extingue la obligacion garantizada cuando asi se pacta, entregando el bien hipotecado en satisfaccion del credito."),
    ("¿Que pasa si firmo sin leer?",
     "El consentimiento contractual prestado por error sustancial puede invalidar el contrato, sin perjuicio de la diligencia exigible a quien contrata."),
    ("¿Puedo grabar a un funcionario que me atiende mal?",
     "El acceso y la captacion de imagenes en espacios y servicios publicos se somete a la normativa de proteccion de datos y al derecho a la propia imagen."),
]
print("pares:", len(PARES))

pares: 30


## 2 · Train y test (preguntas nuevas, mismos fragmentos)

Truco clave para que el fine-tuning **se note de verdad**: entrenamos con TODOS los pares
`(pregunta, fragmento)`, y el test son **paráfrasis NUEVAS** (otra forma de preguntar lo mismo) de
fragmentos que el modelo **sí** afina durante el entrenamiento. Así medimos generalización honesta
(formas de preguntar que no vio) sobre un corpus que sí ha aprendido a representar.

Si el test fueran fragmentos nunca vistos en entrenamiento, con tan pocos datos el modelo no
tendría cómo mejorarlos: por eso antes no se movía el número.


In [3]:
# TRAIN: todos los pares (pregunta, fragmento)
train_pairs = PARES[:]
corpus = [doc for _, doc in PARES]            # los fragmentos = espacio de búsqueda

# TEST: preguntas NUEVAS (paráfrasis en llano) cuyo fragmento correcto está en el corpus.
# (idx = posición en PARES del fragmento correcto). El modelo NO ve estas frases al entrenar.
TEST = [
    ("Mi casero quiere echarme del piso, ¿puede sin mas?",            0),
    ("¿Hasta cuando pueden exigirme pagar algo que debo?",            1),
    ("Si me echan del trabajo, ¿me tienen que indemnizar?",           2),
    ("Heredo de mi padre, que tenia deudas, ¿respondo yo de ellas?",  3),
    ("¿Es legal que mi empresa nos grabe con camaras?",               4),
    ("Compre online algo que no quiero, ¿puedo devolverlo?",          5),
    ("El vecino de arriba no me deja dormir, ¿que puedo hacer?",      7),
    ("¿Puedo dejar toda mi herencia a quien yo quiera?",              8),
    ("Tuve un accidente que no fue culpa mia, ¿quien me paga?",       9),
    ("Quiero montar una empresa con un amigo, ¿que firmamos?",       10),
    ("¿Es legal grabar una conversacion en la que yo participo?",    12),
    ("Mi jefe no me abona las horas extra, ¿como las reclamo?",      13),
    ("Me cobran intereses altisimos por un prestamo, ¿es legal?",    18),
    ("¿Pueden despedir a una empleada embarazada?",                  20),
    ("¿Puedo saldar la hipoteca entregando la casa al banco?",       27),
]
print(f"train: {len(train_pairs)} pares · corpus: {len(corpus)} fragmentos · test: {len(TEST)} preguntas nuevas")

train: 30 pares · corpus: 30 fragmentos · test: 15 preguntas nuevas


## 3 · El modelo base y cómo medimos el retrieval

Cargamos un embedding multilingüe pequeño y definimos la métrica: para cada pregunta de test,
¿el fragmento correcto sale el **primero** (top-1)? ¿y en qué posición media (**MRR**)?


In [4]:
from sentence_transformers import SentenceTransformer, util

BASE_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # ~118M, multilingüe, T4/CPU/MPS
# (en producción elegirías uno más fuerte: intfloat/multilingual-e5-* o BAAI/bge-m3)

def evaluar(model):
    """top-1 accuracy y MRR de las preguntas nuevas (TEST) sobre el corpus completo."""
    corpus_emb = model.encode(corpus, convert_to_tensor=True, normalize_embeddings=True)
    aciertos, rr = 0, 0.0
    for pregunta, idx in TEST:
        q_emb = model.encode(pregunta, convert_to_tensor=True, normalize_embeddings=True)
        scores = util.cos_sim(q_emb, corpus_emb)[0]
        ranking = scores.argsort(descending=True).tolist()
        pos = ranking.index(idx) + 1               # posición del fragmento correcto (1 = primero)
        aciertos += (pos == 1)
        rr += 1.0 / pos
    return aciertos / len(TEST), rr / len(TEST)

In [5]:
model = SentenceTransformer(BASE_MODEL)
acc_base, mrr_base = evaluar(model)
print(f"ANTES (base)  ->  top-1: {acc_base:.0%}   MRR: {mrr_base:.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7708.12it/s]

ANTES (base)  ->  top-1: 53%   MRR: 0.692


## 4 · Fine-tuning contrastivo

`MultipleNegativesRankingLoss` toma pares `(pregunta, fragmento)` y usa el resto del batch como
negativos: acerca cada pregunta a SU fragmento y la aleja de los demás. No necesitas negativos
manuales. Con `BatchSamplers.NO_DUPLICATES` evitamos falsos negativos en el batch.


In [6]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer import losses                    # ruta v6+
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

train_dataset = Dataset.from_dict({
    "anchor":   [q for q, _ in train_pairs],
    "positive": [p for _, p in train_pairs],
})

loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="embed-legal-ft",
    num_train_epochs=20,                 # dataset pequeño -> varias pasadas
    per_device_train_batch_size=16,
    learning_rate=2e-4,
    warmup_steps=0.1,                    # float = ratio de warmup (Transformers v5+)
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    fp16=True,                           # pon False si tu GPU no soporta fp16
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_dataset, loss=loss)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Computing widget examples: 100%|██████████| 1/1 [00:00<00:00,  4.17example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
5,0.728303
10,0.080372
15,0.013960
20,0.026882
25,0.000795
30,0.000825
35,0.000178
40,0.000131


TrainOutput(global_step=40, training_loss=0.1064309257460991, metrics={'train_runtime': 17.7525, 'train_samples_per_second': 33.798, 'train_steps_per_second': 2.253, 'total_flos': 0.0, 'train_loss': 0.1064309257460991, 'epoch': 20.0})

## 5 · Medimos después

El mismo test, con el modelo ya afinado al dominio legal.


In [7]:
acc_ft, mrr_ft = evaluar(model)
print(f"ANTES (base)      ->  top-1: {acc_base:.0%}   MRR: {mrr_base:.3f}")
print(f"DESPUÉS (fine-tune)->  top-1: {acc_ft:.0%}   MRR: {mrr_ft:.3f}")
print(f"\nMejora top-1: {acc_ft-acc_base:+.0%}   ·   mejora MRR: {mrr_ft-mrr_base:+.3f}")

ANTES (base)      ->  top-1: 53%   MRR: 0.692
DESPUÉS (fine-tune)->  top-1: 73%   MRR: 0.825

Mejora top-1: +20%   ·   mejora MRR: +0.134


## 6 · Mirada cualitativa · una consulta en llano

Vemos el top-3 que recupera el modelo afinado para una pregunta de calle.

In [8]:
consulta = "¿Pueden echarme del piso de alquiler sin avisar?"
q_emb = model.encode(consulta, convert_to_tensor=True, normalize_embeddings=True)
corpus_emb = model.encode(corpus, convert_to_tensor=True, normalize_embeddings=True)
scores = util.cos_sim(q_emb, corpus_emb)[0]
top3 = scores.argsort(descending=True).tolist()[:3]
print("Consulta:", consulta, "\n")
for r, idx in enumerate(top3, 1):
    print(f"{r}. ({scores[idx]:.2f}) {corpus[idx][:110]}...")

Consulta: ¿Pueden echarme del piso de alquiler sin avisar? 

1. (0.90) El arrendador no puede instar la resolucion del contrato de arrendamiento sin la notificacion fehaciente y el ...
2. (0.51) El impago de la renta es causa de resolucion del contrato de arrendamiento y habilita el ejercicio de la accio...
3. (0.43) La actualizacion de la renta se rige por lo pactado y por los limites de revision previstos legalmente para lo...


## 7 · Guardar el modelo afinado

Listo para enchufarlo a un vector store (notebook 03) y a un RAG (módulo de RAG).


In [9]:
model.save_pretrained("embed-legal-ft")
print("Guardado en embed-legal-ft/")
# Para subirlo a tu cuenta de HuggingFace:
# from huggingface_hub import login, whoami; login(token=...)
# model.push_to_hub(f"{whoami()['name']}/embed-legal-es")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.78it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.77it/s]

Guardado en embed-legal-ft/


## Cierre · hacia el módulo de RAG

Acabas de mover un número con la palanca más barata del RAG: **afinar el embedding a tu dominio**.

- El embedding genérico no cubría el hueco *lenguaje llano <-> jerga legal*; el fine-tuning sí.
- Mismo patrón que el fine-tuning de un LLM (módulo de Fine-tuning, clase 3), pero sobre el **buscador**, no sobre el generador.
- Con pocos pares y segundos o minutos de entrenamiento, el top-1 sube. Con más datos, más margen.
- Ojo con la varianza: con 15 preguntas de test, cada acierto vale ~7 puntos. En un Mac M-series (MPS) medimos
  top-1 53% -> 73% y MRR 0.692 -> 0.825; en otro hardware (T4, CPU) la cifra exacta puede moverse algún acierto.
  Lo robusto es la tendencia, no el decimal.

**Siguiente (módulo de RAG):** este embedding afinado alimenta el retriever de un RAG. Mejor retrieval,
mejores respuestas.
